# Fine Tuning LLM for Basic Chat Generation

In [1]:
pip install transformers

In [2]:
!pip install -U torchao

In [3]:
import transformers

# Dataset Format:
### {
  'input' : {'role': 'user' , 'mood':['flirty'  ,'friendly', 'serious'], 'text':'xxx'},
'output' : {role': 'bot' , 'mood':['flirty'  ,'friendly', 'serious'], 'text':'xxx'}
}

In [4]:
from google.colab import files
data = files.upload()

Saving WhatsApp Chat with Shiksha Per.txt to WhatsApp Chat with Shiksha Per (2).txt


In [5]:
from datasets import load_dataset
dataset = load_dataset("text", data_files="WhatsApp Chat with Shiksha Per.txt", split ="train")
print(dataset)
print(dataset[0])

Dataset({
    features: ['text'],
    num_rows: 1122
})
{'text': '05/07/24, 3:04\u202fpm - Shiksha Per: Hey'}


In [6]:
merged_chats = []

def clean_chat(data):

    current_sender = None
    current_msgs = []

    for i in range(len(data)):

        line = str(data['text'][i])

        try:
            _, sender_msg = line.split(" - ", 1)
            sender, message = sender_msg.split(":", 1)

            sender = sender.strip()
            message = message.strip()

        except:
            continue

        # same sender continues
        if sender == current_sender:

            current_msgs.append(message)

        # sender changed
        else:

            if current_sender is not None:

                merged_chats.append({
                    "sender": current_sender,
                    "message": "\n".join(current_msgs)
                })

            current_sender = sender
            current_msgs = [message]

    # add final block
    if current_sender is not None:

        merged_chats.append({
            "sender": current_sender,
            "message": "\n".join(current_msgs)
        })

clean_chat(dataset)
from pprint import pprint
pprint(merged_chats[:3])

[{'message': 'Hey\nTune clg liya', 'sender': 'Shiksha Per'},
 {'message': 'Sirt\nSagar institute of research and technology\nTune',
  'sender': 'Priyansh'},
 {'message': 'Admission ho gya\nNhi', 'sender': 'Shiksha Per'}]


In [7]:
pairs = []

for i in range(len(merged_chats) - 1):
    if merged_chats[i]['sender'] == 'Shiksha Per' and merged_chats[i+1]['sender'] =="Priyansh":
        pairs.append({
            "prompt": merged_chats[i]["message"],
            "completion": merged_chats[i + 1]["message"]
        })
pairs[:2]

[{'prompt': 'Hey\nTune clg liya',
  'completion': 'Sirt\nSagar institute of research and technology\nTune'},
 {'prompt': 'Admission ho gya\nNhi', 'completion': 'Ha\nKya karegi'}]

In [27]:
from datasets import Dataset
dataset = Dataset.from_list(pairs)
dataset

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 266
})

In [28]:

len(dataset)

266

In [29]:
dataset[0]

{'prompt': 'Hey\nTune clg liya',
 'completion': 'Sirt\nSagar institute of research and technology\nTune'}

In [11]:
from transformers import AutoTokenizer , AutoModelForCausalLM

In [12]:
model_name = "Qwen/Qwen2.5-1.5B"

In [13]:
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [14]:
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

print(tokenizer.pad_token)
print(tokenizer.pad_token_id)

[PAD]
151665


In [15]:
from transformers import pipeline
llm = pipeline(model=model_name)
llm("Hey dear, how are you?")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "Hey dear, how are you? My name is Julia, and I'm from Croatia. I'm a 12-year-old girl. I'm tall, and my hair is long and black. I have big eyes and a small mouth. I like wearing T-shirts and jeans. I wear glasses because I wear my glasses. Which of the following is NOT true?\nA. She is 12 years old.\nB. She is tall.\nC. She has small eyes.\nD. She likes wearing T-shirts and jeans.\nAnswer: C\n\nWhich of the following statements about the properties of oxygen is correct?\nA. Oxygen has the chemical property of being able to react with all substances\nB. Oxygen has a very low density\nC. Oxygen has a very small volume fraction in the air\nD. Oxygen is a relatively active gas\nAnswer: D\n\nWhich of the following statements about the relationship between force and motion is true?\nA. When the force acting on an object is removed, the object will continue to move at a constant velocity in a straight line.\nB. If an object is subjected to a constant external force, it wi

In [16]:
llm("Or batao , kya haal chaal??")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Or batao , kya haal chaal??!! - Quora\\nOr batao, kya haal chaal??!!\\n4 Answers\\nBest\\nAryan\\n, B.tech Mechanical Engineering, National Institute of Technology, Kurukshetra (2020)\\nAnswered 5 years ago\\nOriginally Answered: Or batao kya haal chaal?\\nHaal chaal hai aur batao ki kya hai haal chaal?\\n5.2K views\\nView upvotes\\nRahul\\n, works at Amazon India\\nAnswered 5 years ago · Author has 138 answers and 1.1M answer views\\nOriginally Answered: Or batao kya haal chaal?\\nHaal chaal hai aur batao ki kya hai haal chaal?\\nHaal chaal hai aur batao ki kya hai haal chaal?\\nHaal chaal hai aur batao ki kya hai haal chaal?\\nHaal chaal hai aur batao ki kya hai haal chaal?\\nHaal chaal hai aur batao ki kya hai haal chaal?\\nHaal chaal hai aur batao'}]

In [30]:

def tokenize(data):
  prompt = data['prompt'] + "\n" + data['completion']
  tokenized = tokenizer(
      prompt,
      truncation = True,
      max_length=32
  )
  return tokenized
from transformers import DataCollatorForLanguageModeling



dataset = dataset.map(tokenize , remove_columns=dataset.column_names)
from pprint import pprint
pprint(dataset[:2])


Map:   0%|          | 0/266 [00:00<?, ? examples/s]

{'attention_mask': [[1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1],
                    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
 'input_ids': [[18665,
                198,
                51,
                2886,
                1185,
                70,
                898,
                7755,
                198,
                50,
                2106,
                198,
                50,
                37712,
                43698,
                315,
                3412,
                323,
               

In [35]:
from peft import get_peft_model , LoraConfig , TaskType

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",],
    lora_dropout=0.005,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model , config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [38]:
from transformers import TrainingArguments , Trainer

train_args = TrainingArguments(
    output_dir="./results",
	per_device_train_batch_size=4,
    learning_rate=0.001,
	num_train_epochs=5,
	logging_steps=10
)

In [39]:


# 2. RE-CREATE the collator so it picks up the change
from transformers import DataCollatorForLanguageModeling
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 3. RE-INITIALIZE the trainer so it uses the new collator
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=dataset,
    data_collator=collator  # The refreshed collator
)

# 4. Now start
trainer.train()


Step,Training Loss
10,9.823843
20,8.236633
30,7.404251
40,6.798245
50,7.178109
60,6.912819
70,6.476479
80,6.034486
90,6.085784
100,6.032338


TrainOutput(global_step=335, training_loss=5.881081088621225, metrics={'train_runtime': 156.0681, 'train_samples_per_second': 8.522, 'train_steps_per_second': 2.146, 'total_flos': 324313112770560.0, 'train_loss': 5.881081088621225, 'epoch': 5.0})

In [40]:
def test_model(user_input):
  inputs = tokenizer(user_input, return_tensors="pt").to('cuda')
  output = model.generate(**inputs, max_length=32,temperature=1.0)
  print(tokenizer.decode(output[0], skip_special_tokens=True))

In [41]:
test_model("Hello , kesi ho?")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Both `max_new_tokens` (=2048) and `max_length`(=32) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hello , kesi ho?1
Kuch
<Media omitted>
<Media omitted>
<Media omitted>
<Media omitted>
<Media omitted>
<Media omitted>
<Media omitted>
Media omitted>
Media omitted>
Media omitted>
<Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
<Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
<Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
Media omitted>
<Media 

In [24]:
test_model("Do you like music? What do you listen to?")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Both `max_new_tokens` (=2048) and `max_length`(=40) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Do you like music? What do you listen to? 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me too 🥲
Me

In [25]:
test_model("kesi ho baby?")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Both `max_new_tokens` (=2048) and `max_length`(=40) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

In [ ]:
test_model("We should really grab dinner somethimes you know 😁")

In [ ]:
test_model("What should i wear for tomorrow's party??")

In [ ]:
test_model("Who is elon musk?")